# Data Cleaning Pipeline — Nepali (Model L)

Based on the **Yi pipeline** (arXiv 2403.04652), adapted for low-resource Nepali:

## Pipeline Stages

| Stage | What it removes | Our adaptation |
| :---: | :--- | :--- |
| 1 | **Normalize** | NFC + wiki/HTML markup removal, URL/PII strip, whitespace collapse |
| 2 | **Language filter** | ≥45% Devanagari for manual, ≥35% for public |
| 3 | **English sentence filter** | Drop sentences with 0 Devanagari chars (pure English nav) |
| 4 | **Heuristic rules** | Length, stopword-aware word repeat, script-based short-line, symbol ratio |
| 5 | **Deduplication** | Exact MD5 (all docs) + MinHash LSH (80+ word docs, threshold=0.80) |

---

### Input
- `../datasets/raw/manual/*.jsonl` — manual collection (~59M words)

### Output
- `../datasets/cleaned/nepali_cleaned_corpus.jsonl`
- Train / Val / Test splits

## 1. Setup & Configuration

In [1]:
import json
import re
import unicodedata
import hashlib
from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm
from datasketch import MinHash, MinHashLSH

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR    = Path("..").resolve()
RAW_DIR     = BASE_DIR / "datasets" / "raw"
CLEANED_DIR = BASE_DIR / "datasets" / "cleaned"
CLEANED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = CLEANED_DIR / "nepali_cleaned_corpus.jsonl"

# ── Regex patterns ─────────────────────────────────────────────────────────────
DEVANAGARI_RE  = re.compile(r'[\u0900-\u097F]')
ASCII_ALPHA_RE = re.compile(r'[a-zA-Z]')
URL_RE         = re.compile(r'https?://\S+|www\.\S+')
EMAIL_RE       = re.compile(r'[\w.+-]+@[\w-]+\.[\w.-]+')
PHONE_RE       = re.compile(r'(?:\+977|977)?[\s-]?\d{9,10}')
MULTI_SPACE_RE = re.compile(r'[ \t]{2,}')
MULTI_NL_RE    = re.compile(r'\n{3,}')
PUNCT_RPT_RE   = re.compile(r'([।!?.]{2,})')
WIKI_MARKUP_RE = re.compile(r'\{\{[^}]*\}\}|\[\[[^\]|]*\|([^\]]*)\]\]|\[\[[^\]]*\]\]|<[^>]*>|==+[^=]+=+=*|\[\d+\]')
SECTION_HDR_RE = re.compile(r'^=+[^=]+=+$', re.MULTILINE)
BULLET_RE      = re.compile(r'^[*#:;]+\s*', re.MULTILINE)

# ── Nepali stopwords (frequent particles/postpositions to exclude from repeat check)
NE_STOPWORDS = {
    'को', 'की', 'का', 'मा', 'र', 'ले', 'लाई', 'ले', 'हो', 'छ', 'थियो', 'हुन्',
    'भयो', 'भएको', 'गरे', 'गरेको', 'गर्ने', 'पनि', 'नि', 'त', 'यो', 'यस',
    'उनी', 'उनले', 'तर', 'अनि', 'कि', 'गरी', 'भन्ने', 'आफ्नो', 'सकेको',
}

# ── Thresholds ─────────────────────────────────────────────────────────────────
class Config:
    # Language filter (relaxed slightly for retention)
    MIN_DEVANAGARI_RATIO_MANUAL = 0.45   # was 0.50
    MIN_DEVANAGARI_RATIO_PUBLIC = 0.35   # was 0.40

    # Length filter
    MIN_WORDS = 8                        # was 10
    MAX_WORDS = 100_000

    # Repetition — stopword-aware (only non-stopwords counted)
    MAX_WORD_REPEAT_RATIO = 0.45         # was 0.40; apply only when doc > 50 words
    MIN_WORDS_FOR_REPEAT  = 50           # was 20

    # Line quality — use script-char presence instead of raw length
    MIN_SCRIPT_CHARS_IN_LINE = 1         # line must have ≥1 Devanagari char to be kept

    # Symbol ratio (count Devanagari as alphanum explicitly)
    MAX_SYMBOL_RATIO = 0.40              # was 0.30

    # Deduplication
    MINHASH_THRESHOLD   = 0.80
    MINHASH_NUM_PERM    = 128
    MINHASH_MIN_WORDS   = 80

cfg = Config()
print("Setup complete.")
print(f"Input:  {RAW_DIR}")
print(f"Output: {OUTPUT_FILE}")


Setup complete.
Input:  /Users/nikhilsai/Documents/Courses/LMA/SelfProject/nepali/datasets/raw
Output: /Users/nikhilsai/Documents/Courses/LMA/SelfProject/nepali/datasets/cleaned/nepali_cleaned_corpus.jsonl


## 2. Cleaning Functions

Each function implements one stage of the pipeline.

In [2]:
def normalize_text(text: str) -> str:
    """Stage 1: Unicode normalization + structured noise removal."""
    # Fix escape chars from JSONL
    text = text.replace("\\n", "\n").replace("\\r", "\r").replace("\\t", "\t")
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # HTML entities
    text = text.replace("&amp;", "&").replace("&nbsp;", " ")
    text = text.replace("&lt;", "<").replace("&gt;", ">")
    text = text.replace("&quot;", '"').replace("&#39;", "'")

    # NFC normalization
    text = unicodedata.normalize('NFC', text)

    # Remove wiki/HTML markup (keep link text where possible)
    text = WIKI_MARKUP_RE.sub(lambda m: m.group(1) if m.lastindex and m.group(1) else '', text)
    text = SECTION_HDR_RE.sub('', text)
    text = BULLET_RE.sub('', text)

    # Remove URLs, PII
    text = URL_RE.sub('', text)
    text = EMAIL_RE.sub('', text)
    text = PHONE_RE.sub('', text)

    # Collapse repeated punctuation (।।।। → ।)
    text = PUNCT_RPT_RE.sub(lambda m: m.group(0)[0], text)

    # Collapse whitespace
    text = MULTI_SPACE_RE.sub(' ', text)
    text = MULTI_NL_RE.sub('\n\n', text)

    # Strip each line
    lines = [line.strip() for line in text.split('\n')]
    text = '\n'.join(line for line in lines if line)

    return text.strip()


def filter_english_sentences(text: str) -> str:
    """Drop sentences that have ZERO Devanagari characters (pure English navigation/boilerplate).
    Keeps mixed sentences (Devanagari + some English terms) intact."""
    sentences = re.split(r'([।॥!?])', text)
    kept = []
    i = 0
    while i < len(sentences):
        s = sentences[i]
        punct = sentences[i + 1] if i + 1 < len(sentences) and re.match(r'[।॥!?]', sentences[i + 1]) else ''
        if punct:
            i += 2
        else:
            i += 1
        full = (s + punct).strip()
        if not full:
            continue
        # Only drop if the sentence has NO Devanagari chars at all
        if len(DEVANAGARI_RE.findall(full)) == 0 and len(full.split()) > 3:
            continue  # pure English nav/boilerplate — skip
        kept.append(full)
    return ' '.join(kept).strip()


def passes_language_filter(text: str, is_manual: bool) -> bool:
    """Stage 2: Check Devanagari script ratio."""
    chars = text.replace(' ', '').replace('\n', '')
    if not chars:
        return False
    deva_count = len(DEVANAGARI_RE.findall(text))
    threshold = cfg.MIN_DEVANAGARI_RATIO_MANUAL if is_manual else cfg.MIN_DEVANAGARI_RATIO_PUBLIC
    return deva_count / len(chars) >= threshold


def passes_heuristic_filter(text: str) -> tuple:
    """Stage 4: Heuristic quality checks. Returns (pass: bool, reason: str)."""
    words = text.split()
    n_words = len(words)

    # Length
    if n_words < cfg.MIN_WORDS:
        return False, "too_short"
    if n_words > cfg.MAX_WORDS:
        return False, "too_long"

    # Stopword-aware word repeat (only for docs with 50+ words)
    if n_words >= cfg.MIN_WORDS_FOR_REPEAT:
        content_words = [w for w in words if w not in NE_STOPWORDS]
        if content_words:
            wc = Counter(content_words)
            max_freq = wc.most_common(1)[0][1] / len(content_words)
            if max_freq > cfg.MAX_WORD_REPEAT_RATIO:
                return False, "word_repeat"

    # Script-based line quality check (replaces crude short-line ratio)
    lines = [l for l in text.split('\n') if l.strip()]
    if len(lines) > 5:
        dead_lines = sum(1 for l in lines if len(DEVANAGARI_RE.findall(l)) == 0 and len(l.split()) > 2)
        if dead_lines / len(lines) > 0.70:
            return False, "no_script_lines"

    # Symbol ratio (count Devanagari explicitly as alphanum)
    chars_no_space = text.replace(' ', '').replace('\n', '')
    if chars_no_space:
        alnum = sum(1 for c in chars_no_space if c.isalnum() or DEVANAGARI_RE.match(c))
        if (1 - alnum / len(chars_no_space)) > cfg.MAX_SYMBOL_RATIO:
            return False, "high_symbol"

    return True, "ok"


def clean_document(text: str, is_manual: bool) -> str | None:
    """Full pipeline for one document. Returns cleaned text or None if filtered."""
    if not text or not text.strip():
        return None

    # Stage 1: Normalize
    text = normalize_text(text)
    if not text:
        return None

    # Stage 2: Language filter
    if not passes_language_filter(text, is_manual):
        return None

    # Stage 3: English sentence filter (manual only)
    if is_manual:
        text = filter_english_sentences(text)
        if not text:
            return None

    # Stage 4: Heuristic filter
    passes, reason = passes_heuristic_filter(text)
    if not passes:
        return None

    return text

print("Cleaning functions defined.")


Cleaning functions defined.


## 3. Deduplication (Exact MD5 + MinHash LSH)

- **Exact MD5**: fast pass for byte-for-byte duplicates across all files
- **MinHash LSH**: near-duplicate detection for docs ≥80 words (threshold=0.80, 128 perms)
- MinHash uses 3-word shingles as in the Yi pipeline

In [3]:
# Exact hash set
exact_hashes = set()

# MinHash LSH
lsh = MinHashLSH(threshold=cfg.MINHASH_THRESHOLD, num_perm=cfg.MINHASH_NUM_PERM)
doc_counter = 0

def create_minhash(text: str) -> MinHash:
    m = MinHash(num_perm=cfg.MINHASH_NUM_PERM)
    words = text.split()
    for i in range(len(words) - 2):
        shingle = f"{words[i]} {words[i+1]} {words[i+2]}"
        m.update(shingle.encode('utf-8'))
    return m

def is_duplicate(text: str) -> tuple:
    """Returns (is_dup: bool, reason: str)."""
    global doc_counter

    # Pass 1: exact match (fast, all docs)
    h = hashlib.md5(text.encode('utf-8')).hexdigest()
    if h in exact_hashes:
        return True, "exact_dup"
    exact_hashes.add(h)

    # Pass 2: MinHash (only 80+ word docs to save memory)
    if len(text.split()) >= cfg.MINHASH_MIN_WORDS:
        m = create_minhash(text)
        if lsh.query(m):
            return True, "near_dup"
        try:
            lsh.insert(f"doc_{doc_counter}", m)
            doc_counter += 1
        except ValueError:
            pass
    return False, ""

print(f"Dedup initialized: exact MD5 + MinHash LSH (threshold={cfg.MINHASH_THRESHOLD}, perms={cfg.MINHASH_NUM_PERM})")


Dedup initialized: exact MD5 + MinHash LSH (threshold=0.8, perms=128)


## 4. Run Pipeline

Process all raw files through the cleaning pipeline.
- Manual sources: stricter (includes English sentence filter)
- Public sources: relaxed language threshold

In [4]:
import time

# Collect input files
manual_files = sorted(f for f in (RAW_DIR / "manual").glob("*.jsonl") if f.stat().st_size > 0)
public_files  = sorted(f for f in (RAW_DIR / "downloaded").glob("*.jsonl") if f.stat().st_size > 0)
print(f"Manual files:  {len(manual_files)}")
print(f"Public files:  {len(public_files)}")

# Stats tracking
stats = {
    "total_input":        0,
    "dropped_language":   0,
    "dropped_heuristic":  0,
    "dropped_exact_dup":  0,
    "dropped_near_dup":   0,
    "retained":           0,
    "retained_words":     0,
}

def process_file(filepath, out_fh, is_manual: bool):
    """Process one JSONL file through the pipeline. Returns (docs_retained, words_retained)."""
    file_docs = file_words = 0
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            stats["total_input"] += 1
            try:
                data  = json.loads(line)
                text  = data.get("text", "")
            except Exception:
                continue

            cleaned = clean_document(text, is_manual)
            if cleaned is None:
                stats["dropped_heuristic"] += 1
                continue

            is_dup, dup_reason = is_duplicate(cleaned)
            if is_dup:
                stats[f"dropped_{dup_reason}"] += 1
                continue

            words = len(cleaned.split())
            out_fh.write(json.dumps({
                "text":   cleaned,
                "source": data.get("source", filepath.stem),
            }, ensure_ascii=False) + "\n")
            stats["retained"]       += 1
            stats["retained_words"] += words
            file_docs  += 1
            file_words += words

    return file_docs, file_words

start = time.time()
print(f"\nOutput: {OUTPUT_FILE}")
print("=" * 70)

with open(OUTPUT_FILE, "w", encoding="utf-8") as out_fh:
    print("\n--- MANUAL SOURCES ---")
    for fp in tqdm(manual_files, desc="Manual files"):
        n_docs, n_words = process_file(fp, out_fh, is_manual=True)
        if n_docs > 0:
            print(f"  {fp.name:<55} {n_docs:>8,} docs  {n_words/1e6:>6.2f}M words")

    if public_files:
        print("\n--- PUBLIC SOURCES ---")
        for fp in tqdm(public_files, desc="Public files"):
            n_docs, n_words = process_file(fp, out_fh, is_manual=False)
            print(f"  {fp.name:<55} {n_docs:>8,} docs  {n_words/1e6:>6.2f}M words")

print(f"\nDone in {(time.time()-start)/60:.1f} min.")
print("=" * 70)


Manual files:  21
Public files:  3

Output: /Users/nikhilsai/Documents/Courses/LMA/SelfProject/nepali/datasets/cleaned/nepali_cleaned_corpus.jsonl

--- MANUAL SOURCES ---


Python(32921) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Manual files:   0%|          | 0/21 [00:00<?, ?it/s]

  nepali_16news.jsonl                                       14,316 docs    4.49M words
  nepali_all_vedas.jsonl                                    16,466 docs    0.78M words
  nepali_bhagwat_mahapuran2.jsonl                           51,599 docs    1.40M words
  nepali_brihat_sabdakosh.jsonl                             79,884 docs    1.19M words
  nepali_classical_books.jsonl                             130,052 docs    4.59M words
  nepali_gorkhapatra.jsonl                                   8,442 docs    0.35M words
  nepali_ia_books.jsonl                                     46,679 docs    5.10M words
  nepali_magazine_corpus.jsonl                              74,475 docs    3.81M words
  nepali_misc_books.jsonl                                   17,484 docs    0.60M words
  nepali_news_ekantipur.jsonl                               38,379 docs    8.76M words
  nepali_news_manual.jsonl                                   6,498 docs    2.82M words
  nepali_news_onlinekhabar.jsonl           

Public files:   0%|          | 0/3 [00:00<?, ?it/s]

  nepali_cc100.jsonl                                       986,746 docs  228.79M words
  nepali_indiccorp.jsonl                                  17,760,685 docs  671.59M words
  nepali_wikipedia.jsonl                                    29,937 docs    4.55M words

Done in 62.2 min.


## 5. Pipeline Summary & Statistics

In [5]:
total_in  = stats["total_input"]
retained  = stats["retained"]
ret_words = stats["retained_words"]

print("=" * 70)
print("CLEANING PIPELINE RESULTS")
print("=" * 70)
print(f"  Total input documents:         {total_in:>12,}")
print(f"  Dropped (heuristic/language):  {stats['dropped_heuristic']:>12,}  ({100*stats['dropped_heuristic']/max(1,total_in):.1f}%)")
print(f"  Dropped (exact duplicates):    {stats['dropped_exact_dup']:>12,}  ({100*stats['dropped_exact_dup']/max(1,total_in):.1f}%)")
print(f"  Dropped (near duplicates):     {stats['dropped_near_dup']:>12,}  ({100*stats['dropped_near_dup']/max(1,total_in):.1f}%)")
print(f"  ──────────────────────────────────────────────────────────────────")
print(f"  RETAINED documents:            {retained:>12,}  ({100*retained/max(1,total_in):.1f}%)")
print(f"  RETAINED words:                {ret_words:>12,}  ({ret_words/1e6:.1f}M)")
print(f"")
print(f"  Output file: {OUTPUT_FILE}")
if OUTPUT_FILE.exists():
    print(f"  Output size: {OUTPUT_FILE.stat().st_size / 1e9:.2f} GB")
print("=" * 70)


CLEANING PIPELINE RESULTS
  Total input documents:           22,498,741
  Dropped (heuristic/language):     2,545,225  (11.3%)
  Dropped (exact duplicates):         551,563  (2.5%)
  Dropped (near duplicates):           58,804  (0.3%)
  ──────────────────────────────────────────────────────────────────
  RETAINED documents:              19,343,149  (86.0%)
  RETAINED words:                 949,472,662  (949.5M)

  Output file: /Users/nikhilsai/Documents/Courses/LMA/SelfProject/nepali/datasets/cleaned/nepali_cleaned_corpus.jsonl
  Output size: 17.15 GB


## 6. Inspect Filtered Documents

Read 20 dropped docs per filter to validate whether the filter is correct.
As per the Yi pipeline lecture: *"Read 20 docs your filter removed. Decide if you agree."*

In [5]:
import random

# Re-run filters on first manual file and collect drops by reason
sample_file = manual_files[0] if manual_files else None
drop_buckets = {"language": [], "heuristic": [], "english_sent": []}

if sample_file:
    with open(sample_file, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 5000:
                break
            if not line.strip():
                continue
            try:
                text = json.loads(line.strip()).get("text", "")
            except Exception:
                continue
            norm = normalize_text(text)
            if not norm:
                continue
            if not passes_language_filter(norm, is_manual=True):
                drop_buckets["language"].append(text[:400])
                continue
            filtered = filter_english_sentences(norm)
            if not filtered:
                drop_buckets["english_sent"].append(norm[:400])
                continue
            passes, reason = passes_heuristic_filter(filtered)
            if not passes:
                drop_buckets.setdefault(reason, []).append(filtered[:400])

    for reason, samples in drop_buckets.items():
        print(f"\n{'='*60}")
        print(f"Filter: {reason} — {len(samples)} dropped in first 5000 docs")
        print('='*60)
        for j, s in enumerate(random.sample(samples, min(5, len(samples))), 1):
            print(f"  [{j}] {s[:250]}")
            print("  " + "-"*40)



Filter: language — 7 dropped in first 5000 docs
  [1] ﻿à¤ªà¥à¤°à¤§à¤¾à¤¨à¤®à¤¨à¥à¤¤à¥à¤°à¥€ à¤•à¥‡à¤ªà¥€ à¤“à¤²à¥€à¤²à¥‡ à¤¨à¥‡à¤ªà¤¾à¤² à¤šà¥€à¤¨à¤•à¥‹ à¤¸à¤‚à¤µà¥‡à¤¦à¤¨à¤¶à¥€à¤²à¤¤à¤¾à¤ªà¥à¤°à¤¤à¤¿ à¤—à¤®à¥à¤­à¥€à¤° à¤°à¤¹à¥‡à¤•à¥‹ à¤­à¤¨à¥à¤¦à¥ˆ à¤¨à¥‡à¤ªà¤¾à¤²à¥€ à¤­à¥‚à¤®à¤¿à¤¬à¤¾à¤Ÿ à¤šà¥€à
  ----------------------------------------
  [2] ﻿à¤šà¥€à¤¨ à¤­à¥à¤°à¤®à¤£à¤®à¤¾ à¤°à¤¹à¥‡à¤•à¤¾ à¤ªà¥à¤°à¤§à¤¾à¤¨à¤®à¤¨à¥à¤¤à¥à¤°à¥€ à¤•à¥‡à¤ªà¥€ à¤“à¤²à¥€à¤²à¥‡ à¤¨à¥‡à¤ªà¤¾à¤²à¤®à¤¾ à¤²à¤—à¤¾à¤¨à¥€à¤•à¤¾ à¤‰à¤ªà¤¯à¥à¤•à¥à¤¤ à¤…à¤µà¤¸à¤°à¤¹à¤°à¥ à¤°à¤¹à¥‡à¤•à¥‹ à¤­à¤¨à¥à¤¦à¥ˆ à¤šà¤¿à¤¨à¤¿
  ----------------------------------------
  [3] ﻿     
(    )  
                                                                                 
                                                      2007.                                        
                                                    
  ----------------------------------------
  [4] ﻿à¤šà¥€à¤¨à¤¸à¤‚à¤— à¤­à¤à¤•à¥‹ à¤ªà¤¾à¤°à¤µ

## 7. Train / Val / Test Split (90/5/5)

In [6]:
import random
random.seed(42)

train_f = CLEANED_DIR / "nepali_train.jsonl"
val_f   = CLEANED_DIR / "nepali_val.jsonl"
test_f  = CLEANED_DIR / "nepali_test.jsonl"
counts  = {"train": [0, 0], "val": [0, 0], "test": [0, 0]}

with open(OUTPUT_FILE) as fin, \
     open(train_f, "w") as ft, \
     open(val_f,   "w") as fv, \
     open(test_f,  "w") as fte:
    for line in fin:
        r = random.random()
        try:
            w = len(json.loads(line).get("text", "").split())
        except Exception:
            w = 0
        if r < 0.90:
            ft.write(line);  counts["train"][0] += 1; counts["train"][1] += w
        elif r < 0.95:
            fv.write(line);  counts["val"][0]   += 1; counts["val"][1]   += w
        else:
            fte.write(line); counts["test"][0]  += 1; counts["test"][1]  += w

print("Train/Val/Test split complete:")
for split, (d, w) in counts.items():
    print(f"  {split:>5}: {d:>10,} docs  {w:>12,} words  ({w/1e6:.1f}M)")


Train/Val/Test split complete:
  train: 17,550,598 docs   904,544,936 words  (904.5M)
    val:    973,596 docs    50,315,211 words  (50.3M)
   test:    975,488 docs    50,152,799 words  (50.2M)
